In [1]:
import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib

from pathlib import Path
from batchgenerators.utilities.file_and_folder_operations import load_json

ModuleNotFoundError: No module named 'batchgenerators'

In [ ]:
checkpoint_path = r"D:\nnUNet\nnUNet_results\Dataset104_ReXGroundingCT\nnUNetTrainer__nnUNetPlans__3d_fullres\fold_0\checkpoint_best.pth"

In [ ]:
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    perform_everything_on_device=True,
    device=device,
    verbose=True,
    verbose_preprocessing=False,
    allow_tqdm=True
)


predictor.initialize_from_trained_model_folder(
    str(Path(checkpoint_path).parent.parent),
    use_folds=(0,)
)


model = predictor.network
model.eval()

print(model)

In [ ]:
for name, module in model.named_modules():
    print(name)

In [ ]:
"""
3D SegGradCAM for a trained nnU-Net v2 model (PlainConvUNet, 3d_fullres).

Fixes vs. the naive version:
  1. Uses nnU-Net's own DefaultPreprocessor -> correct CT normalization + resampling
  2. Runs on a single patch (matching training patch_size), not the whole volume
     -> avoids OOM and shape mismatches, and keeps the comparison patch-scale
     consistent with your LC-KSVD 32^3 patch evaluation
  3. Explicitly loads checkpoint_best.pth (not the default checkpoint_final.pth)
  4. Uses retain_grad() on the forward-hooked tensor instead of a full backward
     hook, which is unreliable with nnU-Net's in-place ReLU blocks
  5. Explicitly disables deep supervision so forward() returns one tensor
"""

import torch
import torch.nn.functional as F
import numpy as np
import nibabel as nib
from pathlib import Path

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.preprocessing.preprocessors.default_preprocessor import DefaultPreprocessor


# ---------------------------------------------------------------------------
# 1. Load the model — explicitly request checkpoint_best.pth
# ---------------------------------------------------------------------------

def load_nnunet_model(trainer_dir: str, fold: int = 0, device: str = None):
    """
    trainer_dir = .../nnUNet_results/DatasetXXX_Name/nnUNetTrainer__nnUNetPlans__3d_fullres
    (the folder that CONTAINS fold_0, fold_1, ..., NOT the fold folder itself)
    """
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=False,   # off for GradCAM — mirroring TTA breaks single-pass attribution
        perform_everything_on_device=True,
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )

    predictor.initialize_from_trained_model_folder(
        trainer_dir,
        use_folds=(fold,),
        checkpoint_name="checkpoint_best.pth",   # <-- was silently defaulting to checkpoint_final.pth
    )

    model = predictor.network
    model.decoder.deep_supervision = False       # ensure forward() returns a single tensor
    model.eval()
    return predictor, model, device


# ---------------------------------------------------------------------------
# 2. Preprocess the CT correctly, using nnU-Net's own pipeline
# ---------------------------------------------------------------------------

def preprocess_ct(predictor: nnUNetPredictor, ct_path: str):
    """
    Returns:
        data: np.ndarray, shape (C, Z, Y, X), correctly normalized + resampled
        properties: dict — needed later if you want to map results back to
                    the original image geometry
    """
    preprocessor = DefaultPreprocessor()
    data, seg, properties = preprocessor.run_case(
        image_files=[ct_path],
        seg_file=None,
        plans_manager=predictor.plans_manager,
        configuration_manager=predictor.configuration_manager,
        dataset_json=predictor.dataset_json,
    )
    return data, properties


# ---------------------------------------------------------------------------
# 3. Extract a single patch at training resolution
# ---------------------------------------------------------------------------

def extract_patch(data: np.ndarray, patch_size, center_zyx=None):
    """
    data: (C, Z, Y, X) preprocessed volume
    patch_size: e.g. (96, 160, 160), same order as data's spatial dims
    center_zyx: voxel coords (z, y, x) to center the patch on (e.g. GT lesion
                centroid). Defaults to the volume center if None.
    """
    c, Z, Y, X = data.shape
    pz, py, px = patch_size

    if center_zyx is None:
        center_zyx = (Z // 2, Y // 2, X // 2)
    cz, cy, cx = center_zyx

    def bounds(center, size, dim_len):
        lo = int(np.clip(center - size // 2, 0, max(dim_len - size, 0)))
        hi = lo + size
        return lo, hi

    z0, z1 = bounds(cz, pz, Z)
    y0, y1 = bounds(cy, py, Y)
    x0, x1 = bounds(cx, px, X)

    patch = data[:, z0:z1, y0:y1, x0:x1]

    # Pad if the volume is smaller than patch_size in any axis
    pad = [(0, 0)]
    for actual, target in zip(patch.shape[1:], patch_size):
        pad.append((0, max(0, target - actual)))
    patch = np.pad(patch, pad, mode="constant", constant_values=0)

    bbox = (z0, z1, y0, y1, x0, x1)
    return patch, bbox


# ---------------------------------------------------------------------------
# 4. 3D SegGradCAM — forward-hook + retain_grad, no backward hook
# ---------------------------------------------------------------------------

class SegGradCAM3D:
    def __init__(self, model: torch.nn.Module, target_layer: torch.nn.Module):
        self.model = model
        self.activation = None
        self._handle = target_layer.register_forward_hook(self._forward_hook)

    def _forward_hook(self, module, inp, out):
        out.retain_grad()          # keep .grad on this non-leaf tensor
        self.activation = out

    def generate(self, volume: torch.Tensor, target_class: int = 1) -> np.ndarray:
        """
        volume: (1, C, D, H, W) tensor, already on the correct device
        Returns a (D, H, W) numpy heatmap, normalized to [0, 1].
        """
        self.model.zero_grad(set_to_none=True)
        output = self.model(volume)              # (1, num_classes, D, H, W)

        assert isinstance(output, torch.Tensor), (
            "model output is not a plain tensor — deep_supervision is probably "
            "still enabled; set model.decoder.deep_supervision = False"
        )

        target_score = output[:, target_class].sum()   # SegGradCAM target (global sum)
        target_score.backward()

        gradients = self.activation.grad          # (1, C, d, h, w) — feature-map resolution
        activations = self.activation.detach()

        weights = gradients.mean(dim=(2, 3, 4), keepdim=True)
        cam = F.relu((weights * activations).sum(dim=1))   # (1, d, h, w)

        cam = cam - cam.amin(dim=(1, 2, 3), keepdim=True)
        cam = cam / cam.amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-8)

        cam = F.interpolate(
            cam.unsqueeze(1), size=volume.shape[2:], mode="trilinear", align_corners=False
        )
        return cam.squeeze().cpu().numpy()

    def close(self):
        self._handle.remove()


# ---------------------------------------------------------------------------
# 5. End-to-end example
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    trainer_dir = r"D:\nnUNet\nnUNet_results\Dataset104_ReXGroundingCT\nnUNetTrainer__nnUNetPlans__3d_fullres"
    ct_path = r"path/to/your_ct.nii.gz"
    patch_size = (96, 160, 160)   # match your actual trained configuration

    predictor, model, device = load_nnunet_model(trainer_dir, fold=0)

    # -- find your target layer once, interactively --
    # for name, _ in model.named_modules(): print(name)
    target_layer = model.decoder.stages[-1]

    data, properties = preprocess_ct(predictor, ct_path)   # (C, Z, Y, X), normalized

    # Center the patch on your GT lesion centroid if you have one, otherwise
    # the volume center is used. For a fair Axis C comparison against
    # LC-KSVD, centering on the same region you evaluate is recommended.
    patch, bbox = extract_patch(data, patch_size)

    volume = torch.from_numpy(patch).float().unsqueeze(0).to(device)  # (1,C,Z,Y,X)

    cam_gen = SegGradCAM3D(model, target_layer)
    cam = cam_gen.generate(volume, target_class=1)   # class 1 = GGO/nodule
    cam_gen.close()

    np.save("nnunet_gradcam_patch.npy", cam)
    print("CAM shape:", cam.shape, "bbox in preprocessed volume:", bbox)

    # For Axis C metrics, resample/crop your ground-truth mask to the same
    # preprocessed-space bbox before computing IoU / Pointing Game / Energy
    # Inside Mask, so both sides of the comparison are in the same grid.